In [ ]:

import logging
import sys

# Configure logging to both file and console
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("training.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def print_log(*args, **kwargs):
    msg = " ".join(map(str, args))
    logging.info(msg)

# Override print to use logging
print = print_log

import logging
import sys

# Configure logging to both file and console
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("training.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def print_log(*args, **kwargs):
    msg = " ".join(map(str, args))
    logging.info(msg)

# Override print to use logging
print = print_log
# Cell 1 — Imports
import os
import copy
import time
import random
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — avoids display errors
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

# torch.amp is the modern API (works for both CPU and CUDA).
# Fall back gracefully on older PyTorch builds.
try:
    from torch.amp import GradScaler, autocast
    _AMP_DEVICE_ARG = True    # new API requires device_type kwarg in autocast
except ImportError:
    from torch.cuda.amp import GradScaler, autocast
    _AMP_DEVICE_ARG = False

import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader

from sklearn.metrics import f1_score, classification_report, confusion_matrix
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print(f'PyTorch      : {torch.__version__}')
print(f'Torchvision  : {torchvision.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')



### ----------------------------------------


In [ ]:

# Cell 3 — Dataset Extraction Check
import zipfile


try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    EXTRACT_DIR = '/content/data'
else:
    EXTRACT_DIR = 'synthetic_industrial_metal_surface_defects\\industrial_defect_dataset'

DONE_FLAG   = os.path.join(EXTRACT_DIR, '.extracted')

# Only attempt to extract if not already done
if not os.path.exists(DONE_FLAG):
    # Look for a ZIP file in common locations
    _zip_candidates = [
        ('/content/drive/MyDrive/synthetic_dataset.zip' if IN_COLAB else 'synthetic_industrial_metal_surface_defects.zip'),
        'synthetic_industrial_metal_surface_defects/synthetic_industrial_metal_surface_defects.zip',
    ]
    ZIP_PATH = next((p for p in _zip_candidates if os.path.exists(p)), None)

    if ZIP_PATH is not None:
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        print(f'Extracting {ZIP_PATH} ...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        with open(DONE_FLAG, 'w') as f:
            f.write('done')
        print('Extraction complete.')
    else:
        # Dataset may already be present without a ZIP — check before raising
        if os.path.isdir(EXTRACT_DIR) and len(os.listdir(EXTRACT_DIR)) > 0:
            print('No ZIP found but EXTRACT_DIR is not empty — assuming data is already present.')
            # Write flag so we skip this block next time
            with open(DONE_FLAG, 'w') as f:
                f.write('done')
        else:
            print('Data already present or ZIP not found. Continuing...')
else:
    print('Already extracted — skipping.')



### ----------------------------------------


In [ ]:

# Cell 4 — Dataset Path Detection
def _find_split_root(root):
    """Walk root and return the first directory that contains both train/ and val/."""
    for dirpath, dirnames, _ in os.walk(root):
        if 'train' in dirnames and 'val' in dirnames:
            return dirpath
    return None

_found = _find_split_root(EXTRACT_DIR)

if _found is None:
    # Print directory tree to aid debugging
    print('Directory tree under EXTRACT_DIR:')
    for dp, dns, fns in os.walk(EXTRACT_DIR):
        depth = dp.replace(EXTRACT_DIR, '').count(os.sep)
        if depth >= 5:
            continue
        print('  ' * depth + os.path.basename(dp) + '/')
    raise FileNotFoundError(
        'Cannot locate train/ and val/ inside the extracted dataset. '
        'Inspect the directory listing above and adjust EXTRACT_DIR if needed.'
    )

DATA_ROOT = _found
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
VAL_DIR   = os.path.join(DATA_ROOT, 'val')

# Validate directories exist
for _name, _path in [('TRAIN_DIR', TRAIN_DIR), ('VAL_DIR', VAL_DIR)]:
    if not os.path.isdir(_path):
        raise FileNotFoundError(f'{_name} does not exist: {_path}')

print(f'DATA_ROOT : {DATA_ROOT}')
print(f'TRAIN_DIR : {TRAIN_DIR}')
print(f'VAL_DIR   : {VAL_DIR}')

# Show per-class image counts
for _split, _path in [('train', TRAIN_DIR), ('val', VAL_DIR)]:
    _cls = sorted(d for d in os.listdir(_path) if os.path.isdir(os.path.join(_path, d)))
    for _c in _cls:
        _n = len([x for x in os.listdir(os.path.join(_path, _c))
                  if not x.startswith('.')])
        print(f'  {_split}/{_c}: {_n} images')



### ----------------------------------------


In [ ]:

# Cell 5 — Configuration
# TRAIN_DIR and VAL_DIR are set automatically by Cell 4 — do not hardcode them here.

SAVE_PATH     = ('/content/model_best.pth' if IN_COLAB else 'model_best.pth')   # local save; copy to Drive manually if needed

CLASS_NAMES   = ['crack', 'hole', 'normal', 'rust', 'scratch']
NUM_CLASSES   = len(CLASS_NAMES)

BATCH_SIZE     = 16      # conservative default; works on both CPU and GPU
NUM_WORKERS    = 0
PIN_MEMORY     = True
TOTAL_EPOCHS   = 10
FREEZE_EPOCHS  = 2       # train head-only for first N epochs
LR_INIT        = 1e-3
DROPOUT_RATE   = 0.4
IMG_SIZE       = 224

# Early stopping
EARLY_STOP_PATIENCE = 4   # stop if val_acc does not improve for this many epochs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device    : {DEVICE}')
print(f'TRAIN_DIR : {TRAIN_DIR}')
print(f'VAL_DIR   : {VAL_DIR}')
print(f'Save path : {SAVE_PATH}')



### ----------------------------------------


In [ ]:

# Cell 7 — Transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # 1-ch grayscale -> 3-ch
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5), transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Transforms defined.')



### ----------------------------------------


In [ ]:

# Cell 2 — Seed Setup (reproducibility)
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f'All seeds set to {SEED} — deterministic mode enabled.')



### ----------------------------------------


In [ ]:

# Cell 6 — Data Loading (FIXED)

from torchvision import datasets

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(root=VAL_DIR,   transform=val_transforms)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Classes:", train_dataset.classes)

# --- STRICT CHECKS ---

# 1. Ensure directories are different
assert TRAIN_DIR != VAL_DIR, "Train and Val paths are SAME!"

# 2. Ensure datasets are not empty
assert len(train_dataset) > 0, "Train dataset is empty!"
assert len(val_dataset) > 0, "Val dataset is empty!"

# 3. Ensure class consistency
assert set(train_dataset.classes) == set(val_dataset.classes), "Class mismatch!"

# 4. CRITICAL: Check overlap
train_files = set([x[0] for x in train_dataset.samples])
val_files   = set([x[0] for x in val_dataset.samples])

overlap = train_files.intersection(val_files)
print("Overlap:", len(overlap))

assert len(overlap) == 0, "DATA LEAKAGE DETECTED!"

print("Dataset validation passed.")



### ----------------------------------------


In [ ]:

# Cell 8 — Dataset Loading & DataLoaders
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(root=VAL_DIR,   transform=val_transforms)

assert len(train_dataset) > 0, 'train_dataset is empty after ImageFolder load.'
assert len(val_dataset)   > 0, 'val_dataset is empty after ImageFolder load.'

print(f'Training samples  : {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Detected classes  : {train_dataset.classes}')

# Confirm class alignment
if train_dataset.classes != CLASS_NAMES:
    raise ValueError(
        f'ImageFolder class order differs from CLASS_NAMES.\n'
        f'  ImageFolder : {train_dataset.classes}\n'
        f'  CLASS_NAMES : {CLASS_NAMES}'
    )

_use_pin = PIN_MEMORY and DEVICE.type == 'cuda'
_use_persistent = NUM_WORKERS > 0

# Worker init function for reproducibility inside DataLoader workers
def _worker_init_fn(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)

_generator = torch.Generator()
_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=_use_pin,
    drop_last=True,
    persistent_workers=_use_persistent,
    worker_init_fn=_worker_init_fn,
    generator=_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=_use_pin,
    persistent_workers=_use_persistent,
    worker_init_fn=_worker_init_fn,
)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')



### ----------------------------------------


In [ ]:

# Cell 9 — Model (EfficientNet-B2, ImageNet pretrained)
def build_model(num_classes: int, dropout: float) -> nn.Module:
    weights = models.EfficientNet_B2_Weights.IMAGENET1K_V1
    m = models.efficientnet_b2(weights=weights)
    in_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return m


model = build_model(NUM_CLASSES, DROPOUT_RATE).to(DEVICE)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_p:,}')
print(f'Trainable params: {trainable_p:,}')



### ----------------------------------------


In [ ]:

# Cell 10 — Loss, Optimizer, Scheduler
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',    # track val accuracy (higher = better)
    factor=0.5,
    patience=2,
)

print('Criterion / Optimizer / Scheduler ready.')
print(f'Initial LR: {LR_INIT}')



### ----------------------------------------


In [ ]:

# Cell 11 — Mixed Precision Setup
USE_AMP = DEVICE.type == 'cuda'

if USE_AMP:
    if _AMP_DEVICE_ARG:
        scaler = GradScaler(device='cuda')
    else:
        scaler = GradScaler()
else:
    scaler = None


def _autocast():
    """Return the correct autocast context manager for the current PyTorch version."""
    if not USE_AMP:
        import contextlib
        return contextlib.nullcontext()
    if _AMP_DEVICE_ARG:
        return autocast(device_type='cuda', dtype=torch.float16)
    return autocast()


print(f'Mixed precision (AMP) enabled : {USE_AMP}')



### ----------------------------------------


In [ ]:

# Cell 12 — Validation Function
def evaluate(model, loader, criterion):
    """
    One full pass over *loader* in eval mode.
    Returns (avg_loss: float, accuracy: float, f1_macro: float).
    """
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with _autocast():
                outputs = model(imgs)
                loss    = criterion(outputs, labels)

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    accuracy = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    f1_mac   = float(f1_score(all_labels, all_preds, average='macro', zero_division=0))
    return avg_loss, accuracy, f1_mac


print('evaluate() ready.')



### ----------------------------------------


In [ ]:

# Cell 13 — Training Loop (with early stopping)
def _set_backbone_grad(model, requires_grad):
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = requires_grad


# ── State ────────────────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_acc  = 0.0
best_f1       = 0.0
best_weights  = None

# Early stopping counters
_es_counter   = 0
_es_best_acc  = 0.0

current_optimizer = optimizer
current_scheduler = scheduler

# ── Loop ─────────────────────────────────────────────────────────────────────
for epoch in range(1, TOTAL_EPOCHS + 1):
    t0 = time.time()

    # ── Freeze / unfreeze backbone ───────────────────────────────────────────
    if epoch == 1:
        _set_backbone_grad(model, False)
        print(f'[Epoch 1-{FREEZE_EPOCHS}] Backbone FROZEN — training head only')

    if epoch == FREEZE_EPOCHS + 1:
        _set_backbone_grad(model, True)
        current_optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR_INIT * 0.1, weight_decay=1e-4,
        )
        current_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            current_optimizer, mode='max', factor=0.5, patience=2
        )
        print(f'[Epoch {epoch}] Backbone UNFROZEN — fine-tuning full model  '
              f'(lr={LR_INIT * 0.1:.1e})')

    # ── Train one epoch ──────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0

    for i, (imgs, labels) in enumerate(train_loader):
        if i % 50 == 0: print(f'  Batch {i}/{len(train_loader)}')
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        current_optimizer.zero_grad(set_to_none=True)

        with _autocast():
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(current_optimizer)
            scaler.update()
        else:
            loss.backward()
            current_optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_dataset)

    # ── Validate ─────────────────────────────────────────────────────────────
    val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion)

    current_scheduler.step(val_acc)

    # ── Checkpoint best model ────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_f1      = val_f1
        best_weights = copy.deepcopy(model.state_dict())
        torch.save(best_weights, SAVE_PATH)
        tag = '  <-- BEST (saved)'
    else:
        tag = ''

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    elapsed = time.time() - t0
    print(
        f'Epoch [{epoch:02d}/{TOTAL_EPOCHS}]  '
        f'loss={train_loss:.4f}  val_loss={val_loss:.4f}  '
        f'val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  '
        f'({elapsed:.0f}s){tag}'
    )

    # ── Early stopping ───────────────────────────────────────────────────────
    if val_acc > _es_best_acc:
        _es_best_acc = val_acc
        _es_counter  = 0
    else:
        _es_counter += 1
        if _es_counter >= EARLY_STOP_PATIENCE:
            print(f'\nEarly stopping triggered at epoch {epoch} '
                  f'(no improvement for {EARLY_STOP_PATIENCE} consecutive epochs).')
            break

print('\nTraining complete.')



### ----------------------------------------


In [ ]:

# Cell 14 — Save & Verify Best Model
if best_weights is None:
    raise RuntimeError('No best_weights found — did the training loop run successfully?')

# Persist (redundant save as safety net in case the loop save was partial)
torch.save(best_weights, SAVE_PATH)
size_mb = os.path.getsize(SAVE_PATH) / (1024 ** 2)
print(f'Model saved  → {SAVE_PATH}  ({size_mb:.1f} MB)')
print(f'Best val accuracy : {best_val_acc * 100:.2f} %')
print(f'Best val F1 macro : {best_f1:.4f}')



### ----------------------------------------


In [ ]:

# Save class names for app.py

import json

CLASS_NAMES = train_dataset.classes

with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)

print("Saved class names:", CLASS_NAMES)



### ----------------------------------------


In [ ]:

# Cell 15 — Reload Saved Model + Sample Inference Verification
# This cell independently verifies the saved checkpoint is valid.

# 1. Rebuild architecture and load weights from disk
loaded_model = build_model(NUM_CLASSES, DROPOUT_RATE)
state_dict   = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=True)
loaded_model.load_state_dict(state_dict)
loaded_model = loaded_model.to(DEVICE)
loaded_model.eval()
print('Checkpoint loaded successfully.')

# 2. Run inference on the first batch from val_loader
sample_imgs, sample_labels = next(iter(val_loader))
sample_imgs = sample_imgs.to(DEVICE)

with torch.no_grad():
    with _autocast():
        sample_outputs = loaded_model(sample_imgs)

sample_probs  = torch.softmax(sample_outputs, dim=1)
sample_preds  = sample_probs.argmax(dim=1).cpu()

# 3. Print first 5 predictions
print('\nSample inference (first 5 images):')
print(f'{"Idx":<5} {"True Label":<15} {"Predicted":<15} {"Confidence":>10}')
print('-' * 50)
for i in range(min(5, len(sample_labels))):
    true_cls = CLASS_NAMES[sample_labels[i]]
    pred_cls = CLASS_NAMES[sample_preds[i]]
    conf     = sample_probs[i, sample_preds[i]].item()
    match    = '✓' if true_cls == pred_cls else '✗'
    print(f'{i:<5} {true_cls:<15} {pred_cls:<15} {conf:>9.2%}  {match}')

print('\nModel reload and sample inference PASSED.')



### ----------------------------------------


In [ ]:

# Cell 16 — Final Evaluation on Best Checkpoint
all_preds_final  = []
all_labels_final = []

loaded_model.eval()
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with _autocast():
            outputs = loaded_model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds_final.extend(preds)
        all_labels_final.extend(labels.numpy())

final_acc = float(np.mean(np.array(all_preds_final) == np.array(all_labels_final)))
final_f1  = float(f1_score(all_labels_final, all_preds_final,
                            average='macro', zero_division=0))

SEP = '=' * 60
print(SEP)
print('               FINAL EVALUATION RESULTS')
print(SEP)
print(f'  Best Val Accuracy  (during training) : {best_val_acc * 100:.2f} %')
print(f'  Final Val Accuracy (best checkpoint) : {final_acc   * 100:.2f} %')
print(f'  Final F1-Score macro                 : {final_f1:.4f}')
print(SEP)
print()
print(classification_report(
    all_labels_final,
    all_preds_final,
    target_names=CLASS_NAMES,
    digits=4,
))



### ----------------------------------------


In [ ]:

# Cell 17 — Training Curves
ep = list(range(1, len(history['train_loss']) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ep, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(ep, history['val_loss'],   'r-o', label='Val Loss')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-entropy loss')
ax1.legend()
ax1.grid(True)

ax2.plot(ep, [v * 100 for v in history['val_acc']], 'g-o', label='Val Accuracy (%)')
ax2.plot(ep, [v * 100 for v in history['val_f1']],  'm-o', label='Val F1 macro (%)')
ax2.set_title('Validation Metrics')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('%')
ax2.legend()
ax2.grid(True)

plt.suptitle(
    f'EfficientNet-B2  |  best acc={best_val_acc*100:.2f}%  f1={best_f1:.4f}',
    fontsize=13
)
plt.tight_layout()

curve_path = 'training_curves.png'
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Training curves saved → {curve_path}')



### ----------------------------------------


In [ ]:


# Cell 18 — Explainable AI (Grad-CAM)
# This section implements Grad-CAM to visualize where the model is looking.
def get_gradcam(model, img_tensor, target_class):
    model.eval()
    # Target the last convolutional block
    target_layer = model.features[-1]
    
    activations = {}
    def get_activations(name):
        def hook(model, input, output): activations[name] = output.detach()
        return hook
    
    gradients = {}
    def get_gradients(name):
        def hook(model, grad_input, grad_output): gradients[name] = grad_output[0].detach()
        return hook
    
    h1 = target_layer.register_forward_hook(get_activations('fwd'))
    h2 = target_layer.register_full_backward_hook(get_gradients('bwd'))
    
    output = model(img_tensor)
    loss = output[0, target_class]
    model.zero_grad()
    loss.backward()
    
    h1.remove(); h2.remove()
    
    weights = torch.mean(gradients['bwd'], dim=(2, 3), keepdim=True)
    cam = torch.sum(weights * activations['fwd'], dim=1, keepdim=True)
    cam = torch.relu(cam)
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.cpu().numpy()[0, 0]

# Cell 19 — Final Visual Quality Audit
# Show 5 random samples with their Grad-CAM heatmaps
plt.figure(figsize=(15, 5))
model.eval()
for i in range(5):
    img, label = val_dataset[random.randint(0, len(val_dataset)-1)]
    input_tensor = img.unsqueeze(0).to(DEVICE)
    output = model(input_tensor)
    pred = output.argmax(1).item()
    
    cam = get_gradcam(model, input_tensor, pred)
    
    plt.subplot(1, 5, i+1)
    # Undo normalization for display
    disp_img = img.permute(1, 2, 0).numpy()
    disp_img = disp_img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    disp_img = np.clip(disp_img, 0, 1)
    
    plt.imshow(disp_img)
    plt.imshow(transforms.ToPILImage()(torch.from_numpy(cam)).resize((224, 224)), alpha=0.4, cmap='jet')
    plt.title(f'T:{CLASS_NAMES[label]}\nP:{CLASS_NAMES[pred]}', color='green' if label==pred else 'red')
    plt.axis('off')
plt.suptitle('Model Heatmaps (Grad-CAM) - Explaining Predictions')
plt.tight_layout()
plt.savefig('explainability_audit.png')
plt.show()

# Cell 20 — Confusion Matrix
cm = confusion_matrix(all_labels_final, all_preds_final)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=ax,
    linewidths=0.5,
    linecolor='lightgrey',
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title(
    f'Confusion Matrix — Val Set  (acc={final_acc*100:.2f}%)',
    fontsize=13
)
plt.tight_layout()

cm_path = 'confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Confusion matrix saved → {cm_path}')
import json
with open("training_history.json", "w") as f:
    json.dump(history, f)
print("Training history saved to training_history.json")
